# NB02: Multi-Species Scale-Up — Manifest, Extraction Handoff, Concatenation

**Goal**: Generate the stratified 24-species manifest (5 phyla × ~5 species), hand off the expensive extraction to `src/extract_multi_species.py`, then concatenate the per-species CSV outputs into the single `multi_species_copy_stats.csv` used by NB03/NB04.

**Why the extraction is externalized**: Per `memories/pitfalls.md`, `jupyter nbconvert --execute` cannot reliably drive a Spark loop that runs for ~80 min — the notebook file is not updated until the whole run finishes, so a mid-run kill (background-task timeout, Spark contention) loses all artifacts. `src/extract_multi_species.py` writes each species result to its own CSV as it goes and is resumable (skips species whose CSV already exists).

**Inputs**:
- `kbase_ke_pangenome.{pangenome, gtdb_species_clade}` for manifest generation (Spark)
- `data/per_species/*.csv` (produced by `src/extract_multi_species.py`)

**Outputs**:
- `data/species_manifest.csv` — full 52-species stratified candidate pool
- `data/species_manifest_25.csv` — reduced 24-species (5 per phylum) manifest actually used
- `data/multi_species_copy_stats.csv` — concatenated per-species output ready for statistical analysis

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
PER_SPECIES_DIR = DATA_DIR / 'per_species'

## 1. Manifest generation (Spark)

Skip this cell if `species_manifest_25.csv` already exists — regenerating it would change the species selection (the pool is deterministic under `random_state=42`, but re-running unnecessarily forces the reviewer to trace two versions).

In [2]:
manifest_path = DATA_DIR / 'species_manifest_25.csv'
if manifest_path.exists():
    print(f'Manifest already exists at {manifest_path}. Skipping regeneration.')
    manifest = pd.read_csv(manifest_path)
else:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
    candidates = spark.sql("""
        SELECT p.gtdb_species_clade_id, p.no_genomes, p.no_gene_clusters, p.no_core,
               sc.GTDB_taxonomy
        FROM kbase_ke_pangenome.pangenome p
        JOIN kbase_ke_pangenome.gtdb_species_clade sc
            ON p.gtdb_species_clade_id = sc.gtdb_species_clade_id
        WHERE p.no_genomes BETWEEN 50 AND 300
    """).toPandas()
    candidates['phylum'] = candidates['GTDB_taxonomy'].str.extract(r'p__([\w]+)')
    # Exclude pilots
    PILOT_PREFIXES = [
        's__Listeria_seeligeri', 's__Stutzerimonas_stutzeri',
        's__Phocaeicola_dorei', 's__Borreliella_burgdorferi',
        's__Bifidobacterium_bifidum',
    ]
    candidates = candidates[~candidates['gtdb_species_clade_id'].str.startswith(tuple(PILOT_PREFIXES))]
    TARGET_PHYLA = ['Pseudomonadota', 'Bacillota', 'Bacteroidota', 'Actinomycetota', 'Campylobacterota']
    selected = []
    for phy in TARGET_PHYLA:
        phy_cand = candidates[candidates['phylum'] == phy].copy()
        phy_cand['sort_key'] = np.abs(phy_cand['no_genomes'] - 130)
        phy_cand = phy_cand.sort_values('sort_key').head(30)
        selected.append(phy_cand.sample(n=min(12, len(phy_cand)), random_state=42))
    full_manifest = pd.concat(selected).reset_index(drop=True)
    full_manifest['species_prefix'] = full_manifest['gtdb_species_clade_id'].str.split('--').str[0]
    full_manifest.to_csv(DATA_DIR / 'species_manifest.csv', index=False)
    manifest = full_manifest.groupby('phylum').head(5).reset_index(drop=True)
    manifest.to_csv(manifest_path, index=False)
    print(f'Wrote {len(manifest)} species to {manifest_path}')

print(f'Manifest: {len(manifest)} species across {manifest.phylum.nunique()} phyla')
print(manifest.groupby('phylum').size().to_string())

Manifest already exists at ../data/species_manifest_25.csv. Skipping regeneration.
Manifest: 24 species across 5 phyla
phylum
Actinomycetota      5
Bacillota           5
Bacteroidota        5
Campylobacterota    4
Pseudomonadota      5


## 2. Extraction handoff

The expensive part — running the 3-way join per species — is done by:

```bash
python projects/gene_copy_number_variation/src/extract_multi_species.py \
    projects/gene_copy_number_variation/data/species_manifest_25.csv \
    projects/gene_copy_number_variation/data/per_species/
```

**Runtime**: ~200 s per species, ~80 min total for 24 species. Resumable — re-running skips species whose CSV already exists.

This cell just confirms that the per-species outputs are present.

In [3]:
expected = set(manifest['species_prefix'].tolist())
present = {f.stem for f in PER_SPECIES_DIR.glob('*.csv')}
missing = expected - present
extra = present - expected
print(f'Expected species: {len(expected)}')
print(f'Per-species CSVs present: {len(present)}')
print(f'Missing: {len(missing)}')
if missing:
    print('  ', sorted(missing))
print(f'Extra (not in manifest): {len(extra)}')
if extra:
    print('  ', sorted(extra))
assert not missing, f'Cannot proceed — run src/extract_multi_species.py first for: {missing}'

Expected species: 24
Per-species CSVs present: 24
Missing: 0
Extra (not in manifest): 0


## 3. Concatenate per-species outputs

Each `data/per_species/{species}.csv` has ~100–160 rows (per COG × per is_core stat). Concatenate into a single frame for downstream analysis.

In [4]:
dfs = [pd.read_csv(f) for f in sorted(PER_SPECIES_DIR.glob('*.csv'))]
combined = pd.concat(dfs, ignore_index=True)
combined.to_csv(DATA_DIR / 'multi_species_copy_stats.csv', index=False)
print(f'Combined: {len(combined):,} rows across {combined.species_prefix.nunique()} species / {combined.phylum.nunique()} phyla')
print(f'Phylum breakdown: {combined.groupby("phylum")["species_prefix"].nunique().to_dict()}')
combined.head()

Combined: 2,768 rows across 24 species / 5 phyla
Phylum breakdown: {'Actinomycetota': 5, 'Bacillota': 5, 'Bacteroidota': 5, 'Campylobacterota': 4, 'Pseudomonadota': 5}


,COG_category,is_core,n_clusters,total_carrier_genomes,total_copies,total_multicopy_genomes,species_prefix,phylum,no_genomes
0,K,False,343,2252,2258,6,s__Aliarcobacter_butzleri,Campylobacterota,81
1,T,False,283,1836,1836,0,s__Aliarcobacter_butzleri,Campylobacterota,81
2,I,False,84,355,366,11,s__Aliarcobacter_butzleri,Campylobacterota,81
3,P,False,308,1972,1976,4,s__Aliarcobacter_butzleri,Campylobacterota,81
4,_missing,True,91,7300,7303,3,s__Aliarcobacter_butzleri,Campylobacterota,81


## 4. Sanity check — COG rate ranking preview

Quick preview that the extracted data reproduces the pilot pattern (L high, H low) before we hand off to NB03 for the formal statistical tests.

In [5]:
per_species_cog = combined.groupby(['species_prefix', 'phylum', 'COG_category']).agg(
    n_clusters=('n_clusters', 'sum'),
    total_carriers=('total_carrier_genomes', 'sum'),
    total_multicopy=('total_multicopy_genomes', 'sum'),
).reset_index()
per_species_cog['rate_pct'] = 100 * per_species_cog['total_multicopy'] / per_species_cog['total_carriers']

single = per_species_cog[per_species_cog['COG_category'].str.len() == 1]
single = single[single['COG_category'] != '-']

cog_rank = single.groupby('COG_category')['rate_pct'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)
print('COG rate ranking (species-level mean weighted rate):')
print(cog_rank.to_string())

print()
print(f'L (adaptive-mobile) mean rate: {cog_rank.loc["L", "mean"]:.3f}%')
print(f'H (housekeeping-coenzyme) mean rate: {cog_rank.loc["H", "mean"]:.3f}%')
print(f'L / H ratio: {cog_rank.loc["L", "mean"] / cog_rank.loc["H", "mean"]:.1f}x')
print('Handoff → NB03 for formal per-species paired tests and figures.')

COG rate ranking (species-level mean weighted rate):
                  mean    median  count
COG_category                           
L             1.572510  1.119621     24
V             0.375399  0.109288     24
S             0.369323  0.341865     24
U             0.342733  0.093237     24
N             0.330451  0.078394     23
K             0.293547  0.122024     24
M             0.248112  0.108073     24
D             0.237189  0.138481     24
G             0.233250  0.112453     24
J             0.168156  0.091822     24
E             0.167822  0.087529     24
Q             0.153559  0.078203     24
O             0.150751  0.081551     24
C             0.144973  0.061559     24
P             0.139978  0.059454     24
T             0.139627  0.094275     24
A             0.135011  0.000000     14
I             0.121801  0.100410     24
F             0.087736  0.061838     24
H             0.081189  0.040896     24
B             0.000000  0.000000     11
Z             0.000000  0.0